# OOBMeanPath: Direct Illustration of the Table 1 Regimes — v3

This notebook empirically illustrates the Table 1 distinction between:

- **fitted reference observations**, for which AllTree and OOBMeanPath can differ on the same fitted forest; and
- **genuinely held-out observations**, for which all fitted trees are eligible and the two scoring rules coincide.

### v3 fix
This version fixes JSON serialization of NumPy scalar types such as `np.int64` in the loader audit. It also retains the compatibility fixes from v2 for SciPy and scikit-learn.

Set `DATA_DIR` in the **Configuration** cell before running the preflight check.


In [1]:
# Table 1 regime illustration for OOBMeanPath
#
# Purpose
# -------
# Empirically contrast the two scoring regimes that are theoretically distinct:
#
#   1) Fitted reference observations:
#      compare ordinary all-tree scoring with OOBMeanPath on the SAME forest.
#
#   2) Genuinely unseen observations:
#      verify that OOB eligibility includes every fitted tree, so OOBMeanPath
#      and ordinary all-tree IF scoring are exactly identical.
#
# The core regime metrics are label-free. Labels are used only for stratified
# train/test splitting and optional retrospective ROC-AUC / AP summaries.
#
# Supported input files: .arff and .mat
#
# Main outputs:
#   table1_regime_seed_level.csv
#   table1_regime_dataset_summary.csv
#   table1_regime_overall_summary.csv
#   table1_regime_loader_audit.csv
#   table1_regime_errors.csv
#   table1_regime_observation_sample.csv
#   table1_regime_reference_rank_shift.pdf
#   table1_regime_Ci_rankshift_association.pdf
#
# Recommended manuscript run:
#   N_REPEATS = 20
#   N_TREES   = 500
#   TEST_SIZE = 0.20
#
# No detector is refitted for the AllTree-vs-OOB comparison within a repeat:
# the same realized forest is used for both scoring rules.

from __future__ import annotations

import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.io import loadmat, arff
from scipy.special import digamma
from scipy.stats import rankdata, spearmanr

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import matplotlib.pyplot as plt

## Configuration

In [2]:
DATA_DIR = Path(".")                  # change to the folder containing .arff/.mat
OUTPUT_DIR = Path("table1_regime_outputs")

EXPECTED_FILE_COUNT = 30              # set to None if the folder count is not fixed
N_REPEATS = 20
N_TREES = 500
TEST_SIZE = 0.20
BASE_SEED = 42

# OOBMeanPath subsample schedule used in the manuscript.
PSI_CAP = 256
PSI_FRACTION = 0.80

# Reference-sample ranking diagnostics.
TOP_FRACTIONS = (0.05, 0.10)
RANK_SHIFT_THRESHOLDS = (0.01, 0.05, 0.10)

# Save at most this many reference observations per dataset from repeat 0.
OBSERVATION_SAMPLE_PER_DATASET = 2000

# Numerical tolerance for a secondary equality audit.
# Exact equality should normally be bitwise because the unseen OOB-eligible
# tree set is the full forest. The tolerance is only a guard for serialization
# or platform-level floating-point differences.
EQUALITY_ATOL = 1e-14

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Utilities

In [3]:
EULER_GAMMA = 0.5772156649015328606


def json_default(obj):
    """Convert common NumPy objects to native Python JSON types."""
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(
        f"Object of type {obj.__class__.__name__} is not JSON serializable"
    )


def c_factor(n):
    """
    Isolation Forest expected unsuccessful-search path length.

    For integer n:
        c(0)=c(1)=0, c(2)=1,
        c(n)=2 H_{n-1} - 2(n-1)/n, n>2.
    """
    x = np.asarray(n, dtype=float)
    out = np.zeros_like(x, dtype=float)

    m2 = x == 2
    out[m2] = 1.0

    mg = x > 2
    if np.any(mg):
        # H_{n-1} = digamma(n) + Euler gamma
        h = digamma(x[mg]) + EULER_GAMMA
        out[mg] = 2.0 * h - 2.0 * (x[mg] - 1.0) / x[mg]

    if np.ndim(n) == 0:
        return float(out)
    return out


def psi_schedule(n: int) -> int:
    if n < 3:
        raise ValueError(f"Need at least 3 reference observations; got n={n}.")
    return int(
        min(
            PSI_CAP,
            max(2, math.floor(PSI_FRACTION * n)),
            n - 1,
        )
    )


def decode_scalar(v):
    if isinstance(v, (bytes, bytearray)):
        return v.decode("utf-8", errors="replace")
    return v


def normalize_label_value(v):
    v = decode_scalar(v)
    if isinstance(v, np.generic):
        v = v.item()
    if isinstance(v, str):
        return v.strip()
    return v


def label_key(v):
    """Stable string form for audit and semantic label matching."""
    v = normalize_label_value(v)
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    return str(v).strip()


def class_counts_dict(y_raw):
    s = pd.Series([label_key(v) for v in np.asarray(y_raw).ravel()])
    return {str(k): int(v) for k, v in s.value_counts(dropna=False).items()}


def binary_anomaly_labels(y_raw, filename: str):
    """
    Convert raw labels to 0=normal, 1=anomaly.

    Rules:
    - Explicit multiclass rules are used for known benchmark files where
      the normal class is defined by the source.
    - For binary semantic labels, anomaly-like text is used when unambiguous.
    - Otherwise, the minority class is treated as anomalous.
    - Unrecognized multiclass labels raise an error rather than guessing.
    """
    raw = np.asarray(y_raw).ravel()
    keys = np.array([label_key(v) for v in raw], dtype=object)
    unique, counts = np.unique(keys, return_counts=True)

    # JSON/pandas friendly audit representation:
    # np.unique returns NumPy scalar types (e.g. np.int64), which the
    # standard-library JSON encoder does not serialize automatically.
    count_map = {
        str(label_key(k)): int(v)
        for k, v in zip(unique, counts)
    }

    fname = filename.lower()

    if len(unique) < 2:
        raise ValueError(
            f"{filename}: target has fewer than two classes: {count_map}"
        )

    # Known multiclass source conventions.
    if "arrhythmia" in fname and len(unique) > 2:
        normal = "1"
        y = (keys != normal).astype(int)
        rule = "Arrhythmia: class 1 normal; all other observed classes anomalous"
        return y, rule, count_map

    if fname == "shuttle.arff" and len(unique) > 2:
        normal = "1"
        y = (keys != normal).astype(int)
        rule = "Shuttle ARFF: class 1 normal; classes 2--7 anomalous"
        return y, rule, count_map

    if len(unique) != 2:
        raise ValueError(
            f"{filename}: multiclass target {count_map} has no explicit rule. "
            "Add a source-specific rule in binary_anomaly_labels()."
        )

    lower = {u: str(u).strip().lower() for u in unique}

    anomaly_tokens = {
        "anomaly", "anomalous", "outlier", "outliers",
        "yes", "true", "positive", "tested_positive",
        "malignant", "m", "attack", "fraud",
    }
    normal_tokens = {
        "normal", "inlier", "inliers",
        "no", "false", "negative", "tested_negative",
        "benign", "b",
    }

    anomaly_hits = [u for u in unique if lower[u] in anomaly_tokens]
    normal_hits = [u for u in unique if lower[u] in normal_tokens]

    if len(anomaly_hits) == 1:
        anomaly_class = anomaly_hits[0]
        rule = f"semantic anomaly label: {anomaly_class}"
    elif len(normal_hits) == 1:
        anomaly_class = [u for u in unique if u != normal_hits[0]][0]
        rule = f"semantic normal label: {normal_hits[0]}; other class anomalous"
    else:
        # Binary fallback: anomaly = minority class.
        anomaly_class = unique[np.argmin(counts)]
        rule = f"binary minority class treated as anomalous: {anomaly_class}"

    y = (keys == anomaly_class).astype(int)

    if y.min() == y.max():
        raise ValueError(f"{filename}: binary conversion failed under rule: {rule}")

    return y, rule, count_map

## Dataset Loading

In [4]:
TARGET_NAMES = {
    "class", "label", "labels", "target", "targets",
    "outlier", "anomaly", "y", "ground_truth", "groundtruth", "gt",
}


def load_arff_dataset(path: Path):
    data, meta = arff.loadarff(path)
    df = pd.DataFrame(data)

    # Decode byte-valued columns.
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].map(decode_scalar)

    lower_to_col = {str(c).strip().lower(): c for c in df.columns}
    target_col = None
    for name in TARGET_NAMES:
        if name in lower_to_col:
            target_col = lower_to_col[name]
            break

    if target_col is None:
        target_col = df.columns[-1]

    y_raw = df[target_col].to_numpy()
    X = df.drop(columns=[target_col]).copy()

    # Replace common textual missing markers.
    X = X.replace({"?": np.nan, "": np.nan})

    return X, y_raw, {
        "feature_source": "ARFF attributes except target",
        "target_source": str(target_col),
    }


def _mat_public_items(obj):
    return {k: v for k, v in obj.items() if not k.startswith("__")}


def _is_vector_like(v):
    if sparse.issparse(v):
        return v.shape[0] == 1 or v.shape[1] == 1
    a = np.asarray(v)
    return a.ndim == 1 or (a.ndim == 2 and 1 in a.shape)


def _vector_length(v):
    if sparse.issparse(v):
        return max(v.shape)
    a = np.asarray(v)
    return a.size


def _flatten_vector(v):
    if sparse.issparse(v):
        return np.asarray(v.toarray()).ravel()
    return np.asarray(v).ravel()


def load_mat_dataset(path: Path):
    obj = _mat_public_items(loadmat(path))

    x_priority = ["X", "x", "data", "Data", "features", "Features"]
    y_priority = [
        "y", "Y", "label", "labels", "Label", "Labels",
        "target", "targets", "Target", "ground_truth", "gt",
    ]

    X = None
    x_name = None
    for k in x_priority:
        if k in obj:
            v = obj[k]
            shape = v.shape if hasattr(v, "shape") else np.asarray(v).shape
            if len(shape) == 2 and min(shape) >= 1:
                X = v
                x_name = k
                break

    # Fallback: largest 2D matrix.
    if X is None:
        candidates = []
        for k, v in obj.items():
            if sparse.issparse(v):
                shape = v.shape
            else:
                a = np.asarray(v)
                shape = a.shape
            if len(shape) == 2 and shape[0] >= 2 and shape[1] >= 1:
                candidates.append((shape[0] * shape[1], k, v))
        if not candidates:
            raise ValueError(f"{path.name}: no 2D feature matrix found in MAT file.")
        _, x_name, X = max(candidates, key=lambda z: z[0])

    n = X.shape[0]

    y = None
    y_name = None
    for k in y_priority:
        if k in obj and _is_vector_like(obj[k]) and _vector_length(obj[k]) == n:
            y = _flatten_vector(obj[k])
            y_name = k
            break

    # Fallback: a vector with length n and a plausible number of classes.
    if y is None:
        candidates = []
        for k, v in obj.items():
            if k == x_name:
                continue
            if _is_vector_like(v) and _vector_length(v) == n:
                flat = _flatten_vector(v)
                try:
                    nunique = pd.Series(flat).nunique(dropna=False)
                except Exception:
                    continue
                if 2 <= nunique <= max(50, int(np.sqrt(n)) + 2):
                    candidates.append((nunique, k, flat))
        if not candidates:
            raise ValueError(
                f"{path.name}: no target vector of length n={n} found in MAT file."
            )
        _, y_name, y = min(candidates, key=lambda z: z[0])

    # If feature matrix was stored transposed and y reveals the intended n.
    if len(y) != X.shape[0] and len(y) == X.shape[1]:
        X = X.T
        n = X.shape[0]

    if len(y) != n:
        raise ValueError(
            f"{path.name}: feature rows={n}, target length={len(y)}."
        )

    return X, y, {
        "feature_source": f"MAT key {x_name}",
        "target_source": f"MAT key {y_name}",
    }


def load_dataset(path: Path):
    suffix = path.suffix.lower()
    if suffix == ".arff":
        X, y_raw, info = load_arff_dataset(path)
    elif suffix == ".mat":
        X, y_raw, info = load_mat_dataset(path)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    y, rule, counts = binary_anomaly_labels(y_raw, path.name)

    if len(y) != X.shape[0]:
        raise ValueError(
            f"{path.name}: X has {X.shape[0]} rows but y has {len(y)} entries."
        )

    audit = {
        "dataset": path.name,
        "n": int(len(y)),
        "raw_class_counts": json.dumps(counts, sort_keys=True, default=json_default),
        "anomaly_rule": rule,
        "n_anomalies": int(y.sum()),
        "anomaly_fraction": float(y.mean()),
        **info,
    }
    return X, y, audit

## Preprocessing

In [5]:
def fit_transform_reference_heldout(X_ref_raw, X_test_raw):
    """
    Fit preprocessing on the reference split only, then transform held-out data.
    """
    if sparse.issparse(X_ref_raw):
        # MAT numeric sparse path.
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler(with_mean=False)
        pipe = Pipeline([
            ("imputer", imputer),
            ("scaler", scaler),
        ])
        X_ref = pipe.fit_transform(X_ref_raw).tocsr().astype(np.float32, copy=False)
        X_test = pipe.transform(X_test_raw).tocsr().astype(np.float32, copy=False)
        X_ref.sort_indices()
        X_test.sort_indices()
        return X_ref, X_test, pipe

    if isinstance(X_ref_raw, np.ndarray):
        # MAT numeric dense path.
        X_ref_df = pd.DataFrame(X_ref_raw)
        X_test_df = pd.DataFrame(X_test_raw, columns=X_ref_df.columns)
    else:
        X_ref_df = X_ref_raw.copy()
        X_test_df = X_test_raw.copy()

    numeric_cols = [
        c for c in X_ref_df.columns
        if pd.api.types.is_numeric_dtype(X_ref_df[c])
    ]
    categorical_cols = [c for c in X_ref_df.columns if c not in numeric_cols]

    transformers = []

    if numeric_cols:
        numeric_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])
        transformers.append(("num", numeric_pipe, numeric_cols))

    if categorical_cols:
        categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
        transformers.append(("cat", categorical_pipe, categorical_cols))

    if not transformers:
        raise ValueError("No usable feature columns after preprocessing audit.")

    pre = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.30,
    )

    X_ref = pre.fit_transform(X_ref_df)
    X_test = pre.transform(X_test_df)

    # IsolationForest accepts dense arrays and sparse matrices.
    if sparse.issparse(X_ref):
        # Individual ExtraTreeRegressor methods are stricter than the
        # IsolationForest wrapper in some sklearn versions.  Use CSR,
        # float32, and sorted indices explicitly.
        X_ref = X_ref.tocsr().astype(np.float32, copy=False)
        X_test = X_test.tocsr().astype(np.float32, copy=False)
        X_ref.sort_indices()
        X_test.sort_indices()
    else:
        X_ref = np.asarray(X_ref, dtype=np.float32)
        X_test = np.asarray(X_test, dtype=np.float32)

    return X_ref, X_test, pre

## Same-Forest Path Scoring

In [6]:
def tree_completed_path_length(estimator, X):
    """
    Completed IF path length for every row of X in one fitted tree.

    depth = number of traversed edges
    completed path = depth + c(number of training samples in terminal leaf)
    """
    # Keep compatibility with sklearn versions whose individual tree
    # methods require float32 input even when IsolationForest.fit()
    # accepted float64 input.
    if sparse.issparse(X):
        X_tree = X.tocsr().astype(np.float32, copy=False)
        X_tree.sort_indices()
    else:
        X_tree = np.asarray(X, dtype=np.float32)

    node_indicator = estimator.decision_path(X_tree)
    depth = np.asarray(node_indicator.sum(axis=1)).ravel().astype(float) - 1.0

    leaves = estimator.apply(X_tree)
    n_leaf = estimator.tree_.n_node_samples[leaves].astype(float)

    return depth + c_factor(n_leaf)


def fit_and_score_same_forest(X_ref, X_test, seed: int):
    n_ref = X_ref.shape[0]
    psi = psi_schedule(n_ref)
    q = 1.0 - psi / n_ref

    forest = IsolationForest(
        n_estimators=N_TREES,
        max_samples=psi,
        contamination="auto",
        max_features=1.0,
        bootstrap=False,
        n_jobs=-1,
        random_state=int(seed),
    )
    forest.fit(X_ref)

    ref_all_sum = np.zeros(n_ref, dtype=np.float64)
    ref_oob_sum = np.zeros(n_ref, dtype=np.float64)
    ref_oob_count = np.zeros(n_ref, dtype=np.int32)

    n_test = X_test.shape[0]
    test_all_sum = np.zeros(n_test, dtype=np.float64)

    # This accumulator follows the OOB eligibility rule for unseen queries.
    # Because an unseen query was in no construction subsample, every tree
    # is eligible. It is updated independently from test_all_sum as an audit.
    test_oob_equiv_sum = np.zeros(n_test, dtype=np.float64)
    test_oob_equiv_count = np.zeros(n_test, dtype=np.int32)

    sample_index_lists = forest.estimators_samples_

    for tree, inbag_idx in zip(forest.estimators_, sample_index_lists):
        h_ref = tree_completed_path_length(tree, X_ref)
        ref_all_sum += h_ref

        eligible = np.ones(n_ref, dtype=bool)
        eligible[np.asarray(inbag_idx, dtype=int)] = False
        ref_oob_sum[eligible] += h_ref[eligible]
        ref_oob_count[eligible] += 1

        h_test = tree_completed_path_length(tree, X_test)
        test_all_sum += h_test

        # For unseen queries: I_*t = 0 for all t, so all trees are eligible.
        test_oob_equiv_sum += h_test
        test_oob_equiv_count += 1

    if np.any(ref_oob_count == 0):
        bad = int(np.sum(ref_oob_count == 0))
        raise RuntimeError(
            f"{bad} reference observations received zero OOB trees. "
            "Increase N_TREES."
        )

    ref_mean_all = ref_all_sum / N_TREES
    ref_mean_oob = ref_oob_sum / ref_oob_count

    test_mean_all = test_all_sum / N_TREES
    test_mean_oob_equiv = test_oob_equiv_sum / test_oob_equiv_count

    cpsi = c_factor(psi)

    ref_score_all = np.power(2.0, -ref_mean_all / cpsi)
    ref_score_oob = np.power(2.0, -ref_mean_oob / cpsi)

    test_score_all = np.power(2.0, -test_mean_all / cpsi)
    test_score_oob_equiv = np.power(2.0, -test_mean_oob_equiv / cpsi)

    C = ref_mean_all - ref_mean_oob

    return {
        "forest": forest,
        "psi": psi,
        "q": q,
        "cpsi": cpsi,
        "ref_oob_count": ref_oob_count,
        "ref_mean_all": ref_mean_all,
        "ref_mean_oob": ref_mean_oob,
        "ref_score_all": ref_score_all,
        "ref_score_oob": ref_score_oob,
        "C": C,
        "test_mean_all": test_mean_all,
        "test_mean_oob_equiv": test_mean_oob_equiv,
        "test_score_all": test_score_all,
        "test_score_oob_equiv": test_score_oob_equiv,
    }

## Metrics

In [7]:
def anomaly_percentile_ranks(scores):
    """
    0 = most anomalous end, 1 = least anomalous end.
    """
    n = len(scores)
    if n <= 1:
        return np.zeros(n, dtype=float)
    r = rankdata(-np.asarray(scores), method="average")
    return (r - 1.0) / (n - 1.0)


def top_overlap(scores_a, scores_b, frac):
    n = len(scores_a)
    k = max(1, int(math.ceil(frac * n)))
    ia = set(np.argsort(-np.asarray(scores_a), kind="mergesort")[:k].tolist())
    ib = set(np.argsort(-np.asarray(scores_b), kind="mergesort")[:k].tolist())
    return len(ia & ib) / k


def safe_spearman(x, y):
    """
    SciPy-version-compatible Spearman correlation.
    Newer SciPy uses result.statistic; older versions use
    result.correlation. Tuple indexing works as a final fallback.
    """
    x = np.asarray(x)
    y = np.asarray(y)

    if len(x) < 3:
        return np.nan
    if np.nanstd(x) == 0 or np.nanstd(y) == 0:
        return np.nan

    result = spearmanr(x, y, nan_policy="omit")

    if hasattr(result, "statistic"):
        value = result.statistic
    elif hasattr(result, "correlation"):
        value = result.correlation
    else:
        value = result[0]

    return float(value)


def safe_auc(y, score):
    y = np.asarray(y)
    if len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, score))


def safe_ap(y, score):
    y = np.asarray(y)
    if len(np.unique(y)) < 2:
        return np.nan
    return float(average_precision_score(y, score))


def split_raw(X, y, seed):
    """
    Stratified 80/20 split at the raw-feature level.
    """
    indices = np.arange(len(y))

    try:
        ref_idx, test_idx = train_test_split(
            indices,
            test_size=TEST_SIZE,
            random_state=int(seed),
            stratify=y,
        )
    except ValueError as exc:
        raise ValueError(
            f"Stratified split failed: {exc}. "
            "Check that each class has enough observations."
        )

    if len(np.unique(y[ref_idx])) < 2 or len(np.unique(y[test_idx])) < 2:
        raise ValueError(
            "Reference or held-out split has fewer than two classes."
        )

    if sparse.issparse(X):
        X_ref_raw = X[ref_idx]
        X_test_raw = X[test_idx]
    elif isinstance(X, pd.DataFrame):
        X_ref_raw = X.iloc[ref_idx].reset_index(drop=True)
        X_test_raw = X.iloc[test_idx].reset_index(drop=True)
    else:
        X_ref_raw = np.asarray(X)[ref_idx]
        X_test_raw = np.asarray(X)[test_idx]

    return X_ref_raw, X_test_raw, y[ref_idx], y[test_idx], ref_idx, test_idx

## One Repeat

In [8]:
def run_one_repeat(dataset_name, X_raw, y, seed, repeat_index):
    (
        X_ref_raw,
        X_test_raw,
        y_ref,
        y_test,
        ref_original_idx,
        test_original_idx,
    ) = split_raw(X_raw, y, seed)

    X_ref, X_test, pre = fit_transform_reference_heldout(
        X_ref_raw, X_test_raw
    )

    out = fit_and_score_same_forest(X_ref, X_test, seed)

    rank_all = anomaly_percentile_ranks(out["ref_score_all"])
    rank_oob = anomaly_percentile_ranks(out["ref_score_oob"])
    abs_rank_shift = np.abs(rank_oob - rank_all)

    C_norm_abs = np.abs(out["C"]) / out["cpsi"]

    ref_score_diff = out["ref_score_oob"] - out["ref_score_all"]
    test_path_diff = out["test_mean_oob_equiv"] - out["test_mean_all"]
    test_score_diff = out["test_score_oob_equiv"] - out["test_score_all"]

    row = {
        "dataset": dataset_name,
        "repeat": int(repeat_index),
        "seed": int(seed),
        "n_total": int(len(y)),
        "n_reference": int(len(y_ref)),
        "n_heldout": int(len(y_test)),
        "d_processed": int(X_ref.shape[1]),
        "psi": int(out["psi"]),
        "q_oob": float(out["q"]),
        "expected_oob_trees": float(N_TREES * out["q"]),
        "mean_realized_oob_trees": float(np.mean(out["ref_oob_count"])),
        "min_realized_oob_trees": int(np.min(out["ref_oob_count"])),
        "max_realized_oob_trees": int(np.max(out["ref_oob_count"])),

        # Core label-free reference-regime diagnostics.
        "reference_score_spearman": safe_spearman(
            out["ref_score_all"], out["ref_score_oob"]
        ),
        "reference_mean_abs_score_diff": float(np.mean(np.abs(ref_score_diff))),
        "reference_median_abs_score_diff": float(np.median(np.abs(ref_score_diff))),
        "reference_max_abs_score_diff": float(np.max(np.abs(ref_score_diff))),
        "reference_mean_abs_rank_shift": float(np.mean(abs_rank_shift)),
        "reference_median_abs_rank_shift": float(np.median(abs_rank_shift)),
        "reference_p95_abs_rank_shift": float(np.quantile(abs_rank_shift, 0.95)),
        "reference_max_abs_rank_shift": float(np.max(abs_rank_shift)),
        "reference_median_abs_C_over_cpsi": float(np.median(C_norm_abs)),
        "reference_p95_abs_C_over_cpsi": float(np.quantile(C_norm_abs, 0.95)),
        "reference_rho_absC_rankshift": safe_spearman(
            C_norm_abs, abs_rank_shift
        ),

        # Unseen-query equality audit.
        "heldout_max_abs_path_diff": float(np.max(np.abs(test_path_diff))),
        "heldout_max_abs_score_diff": float(np.max(np.abs(test_score_diff))),
        "heldout_mean_abs_score_diff": float(np.mean(np.abs(test_score_diff))),
        "heldout_exact_path_equal": bool(
            np.array_equal(out["test_mean_oob_equiv"], out["test_mean_all"])
        ),
        "heldout_exact_score_equal": bool(
            np.array_equal(out["test_score_oob_equiv"], out["test_score_all"])
        ),
        "heldout_equal_within_tolerance": bool(
            np.max(np.abs(test_score_diff)) <= EQUALITY_ATOL
        ),

        # Optional retrospective label-based summaries.
        "reference_ROC_all": safe_auc(y_ref, out["ref_score_all"]),
        "reference_ROC_oob": safe_auc(y_ref, out["ref_score_oob"]),
        "reference_AP_all": safe_ap(y_ref, out["ref_score_all"]),
        "reference_AP_oob": safe_ap(y_ref, out["ref_score_oob"]),
        "heldout_ROC_standard": safe_auc(y_test, out["test_score_all"]),
        "heldout_ROC_oob_equiv": safe_auc(y_test, out["test_score_oob_equiv"]),
        "heldout_AP_standard": safe_ap(y_test, out["test_score_all"]),
        "heldout_AP_oob_equiv": safe_ap(y_test, out["test_score_oob_equiv"]),
    }

    for frac in TOP_FRACTIONS:
        pct = int(round(100 * frac))
        row[f"reference_top{pct}_overlap"] = top_overlap(
            out["ref_score_all"], out["ref_score_oob"], frac
        )
        row[f"heldout_top{pct}_overlap"] = top_overlap(
            out["test_score_all"], out["test_score_oob_equiv"], frac
        )

    for thr in RANK_SHIFT_THRESHOLDS:
        pct = int(round(100 * thr))
        row[f"reference_fraction_rank_shift_ge_{pct}pct"] = float(
            np.mean(abs_rank_shift >= thr)
        )

    row["reference_delta_ROC"] = row["reference_ROC_oob"] - row["reference_ROC_all"]
    row["reference_delta_AP"] = row["reference_AP_oob"] - row["reference_AP_all"]
    row["heldout_delta_ROC"] = (
        row["heldout_ROC_oob_equiv"] - row["heldout_ROC_standard"]
    )
    row["heldout_delta_AP"] = (
        row["heldout_AP_oob_equiv"] - row["heldout_AP_standard"]
    )

    observation_sample = None

    if repeat_index == 0:
        n_ref = len(y_ref)
        if n_ref <= OBSERVATION_SAMPLE_PER_DATASET:
            keep = np.arange(n_ref)
        else:
            rng = np.random.default_rng(seed + 99173)
            keep = np.sort(
                rng.choice(
                    n_ref,
                    size=OBSERVATION_SAMPLE_PER_DATASET,
                    replace=False,
                )
            )

        observation_sample = pd.DataFrame({
            "dataset": dataset_name,
            "repeat": repeat_index,
            "seed": int(seed),
            "reference_row_within_split": keep,
            "original_row_index": ref_original_idx[keep],
            "label_for_retrospective_evaluation": y_ref[keep],
            "score_all": out["ref_score_all"][keep],
            "score_oob": out["ref_score_oob"][keep],
            "mean_path_all": out["ref_mean_all"][keep],
            "mean_path_oob": out["ref_mean_oob"][keep],
            "C_i": out["C"][keep],
            "abs_C_over_cpsi": C_norm_abs[keep],
            "rank_percentile_all": rank_all[keep],
            "rank_percentile_oob": rank_oob[keep],
            "abs_rank_shift": abs_rank_shift[keep],
            "oob_tree_count": out["ref_oob_count"][keep],
        })

    return row, observation_sample

## Summaries

In [9]:
def dataset_summary(seed_df):
    rows = []

    for dataset, g in seed_df.groupby("dataset", sort=True):
        row = {
            "dataset": dataset,
            "repeats": int(len(g)),
            "n_total": int(g["n_total"].iloc[0]),
            "n_reference_mean": float(g["n_reference"].mean()),
            "n_heldout_mean": float(g["n_heldout"].mean()),
            "d_processed_mean": float(g["d_processed"].mean()),
            "psi_mean": float(g["psi"].mean()),
            "q_oob_mean": float(g["q_oob"].mean()),

            "reference_score_spearman_mean": float(g["reference_score_spearman"].mean()),
            "reference_mean_abs_rank_shift_mean": float(
                g["reference_mean_abs_rank_shift"].mean()
            ),
            "reference_median_abs_rank_shift_mean": float(
                g["reference_median_abs_rank_shift"].mean()
            ),
            "reference_p95_abs_rank_shift_mean": float(
                g["reference_p95_abs_rank_shift"].mean()
            ),
            "reference_fraction_rank_shift_ge_5pct_mean": float(
                g["reference_fraction_rank_shift_ge_5pct"].mean()
            ),
            "reference_top5_overlap_mean": float(g["reference_top5_overlap"].mean()),
            "reference_top10_overlap_mean": float(g["reference_top10_overlap"].mean()),
            "reference_rho_absC_rankshift_mean": float(
                g["reference_rho_absC_rankshift"].mean()
            ),
            "reference_median_abs_C_over_cpsi_mean": float(
                g["reference_median_abs_C_over_cpsi"].mean()
            ),
            "reference_p95_abs_C_over_cpsi_mean": float(
                g["reference_p95_abs_C_over_cpsi"].mean()
            ),

            "heldout_max_abs_path_diff_max": float(
                g["heldout_max_abs_path_diff"].max()
            ),
            "heldout_max_abs_score_diff_max": float(
                g["heldout_max_abs_score_diff"].max()
            ),
            "heldout_exact_path_equal_all_repeats": bool(
                g["heldout_exact_path_equal"].all()
            ),
            "heldout_exact_score_equal_all_repeats": bool(
                g["heldout_exact_score_equal"].all()
            ),
            "heldout_equal_within_tolerance_all_repeats": bool(
                g["heldout_equal_within_tolerance"].all()
            ),

            # Descriptive only.
            "reference_delta_ROC_mean": float(g["reference_delta_ROC"].mean()),
            "reference_delta_AP_mean": float(g["reference_delta_AP"].mean()),
            "heldout_delta_ROC_max_abs": float(np.max(np.abs(g["heldout_delta_ROC"]))),
            "heldout_delta_AP_max_abs": float(np.max(np.abs(g["heldout_delta_AP"]))),
        }

        rows.append(row)

    return pd.DataFrame(rows)


def overall_summary(seed_df, ds_df):
    return pd.DataFrame([{
        "n_datasets": int(ds_df.shape[0]),
        "n_dataset_repeat_runs": int(seed_df.shape[0]),
        "n_trees": int(N_TREES),
        "n_repeats": int(N_REPEATS),
        "test_size": float(TEST_SIZE),

        "reference_dataset_median_of_mean_abs_rank_shift": float(
            ds_df["reference_mean_abs_rank_shift_mean"].median()
        ),
        "reference_dataset_mean_of_mean_abs_rank_shift": float(
            ds_df["reference_mean_abs_rank_shift_mean"].mean()
        ),
        "reference_dataset_median_top10_overlap": float(
            ds_df["reference_top10_overlap_mean"].median()
        ),
        "reference_dataset_median_fraction_rank_shift_ge_5pct": float(
            ds_df["reference_fraction_rank_shift_ge_5pct_mean"].median()
        ),
        "reference_dataset_median_rho_absC_rankshift": float(
            ds_df["reference_rho_absC_rankshift_mean"].median()
        ),

        "heldout_global_max_abs_path_diff": float(
            seed_df["heldout_max_abs_path_diff"].max()
        ),
        "heldout_global_max_abs_score_diff": float(
            seed_df["heldout_max_abs_score_diff"].max()
        ),
        "heldout_exact_path_equal_all_runs": bool(
            seed_df["heldout_exact_path_equal"].all()
        ),
        "heldout_exact_score_equal_all_runs": bool(
            seed_df["heldout_exact_score_equal"].all()
        ),
        "heldout_equal_within_tolerance_all_runs": bool(
            seed_df["heldout_equal_within_tolerance"].all()
        ),

        # Descriptive only; not the basis for Table 1.
        "reference_delta_ROC_across_datasets_mean": float(
            ds_df["reference_delta_ROC_mean"].mean()
        ),
        "reference_delta_AP_across_datasets_mean": float(
            ds_df["reference_delta_AP_mean"].mean()
        ),
    }])

## Figures

In [10]:
def make_figures(ds_df):
    plot_df = ds_df.sort_values(
        "reference_mean_abs_rank_shift_mean",
        ascending=True,
    ).copy()

    fig, ax = plt.subplots(figsize=(9.5, 10.0))
    y = np.arange(len(plot_df))
    ax.scatter(
        plot_df["reference_mean_abs_rank_shift_mean"],
        y,
        s=30,
    )
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df["dataset"], fontsize=7)
    ax.set_xlabel("Mean absolute percentile-rank shift: OOBMeanPath vs AllTree")
    ax.set_ylabel("Dataset")
    ax.set_title("Reference-sample ranking changes on the same fitted forest")
    fig.subplots_adjust(left=0.35, right=0.98, top=0.94, bottom=0.08)
    fig.savefig(
        OUTPUT_DIR / "table1_regime_reference_rank_shift.pdf",
        bbox_inches="tight",
    )
    fig.savefig(
        OUTPUT_DIR / "table1_regime_reference_rank_shift.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8.2, 6.2))
    ax.scatter(
        ds_df["reference_p95_abs_C_over_cpsi_mean"],
        ds_df["reference_mean_abs_rank_shift_mean"],
        s=34,
    )
    ax.set_xlabel(r"Mean across repeats of 95th percentile $|C_i|/c(\psi)$")
    ax.set_ylabel("Mean absolute percentile-rank shift")
    ax.set_title("Self-inclusion sensitivity and reference-rank movement")
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "table1_regime_Ci_rankshift_association.pdf",
        bbox_inches="tight",
    )
    fig.savefig(
        OUTPUT_DIR / "table1_regime_Ci_rankshift_association.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.close(fig)

## Main

In [11]:
def main():
    import scipy
    import sklearn

    print("Environment:")
    print(f"  NumPy:        {np.__version__}")
    print(f"  pandas:       {pd.__version__}")
    print(f"  SciPy:        {scipy.__version__}")
    print(f"  scikit-learn: {sklearn.__version__}")
    print()

    files = sorted(
        [
            p for p in DATA_DIR.iterdir()
            if p.is_file() and p.suffix.lower() in {".arff", ".mat"}
        ],
        key=lambda p: p.name.lower(),
    )

    if EXPECTED_FILE_COUNT is not None and len(files) != EXPECTED_FILE_COUNT:
        warnings.warn(
            f"Found {len(files)} .arff/.mat files; "
            f"EXPECTED_FILE_COUNT={EXPECTED_FILE_COUNT}."
        )

    if not files:
        raise RuntimeError(
            f"No .arff or .mat files found in {DATA_DIR.resolve()}."
        )

    seed_state = np.random.SeedSequence(BASE_SEED).generate_state(N_REPEATS)
    repeat_seeds = [int(x) for x in seed_state]

    loader_audit = []
    run_rows = []
    observation_parts = []
    errors = []

    print(f"Discovered {len(files)} dataset files.")
    print(f"Repeats per dataset: {N_REPEATS}")
    print(f"Trees per forest: {N_TREES}")
    print()

    for file_index, path in enumerate(files, start=1):
        print(f"[{file_index:02d}/{len(files):02d}] {path.name}")

        try:
            X_raw, y, audit = load_dataset(path)
            loader_audit.append(audit)
        except Exception as exc:
            errors.append({
                "dataset": path.name,
                "stage": "load",
                "error_type": type(exc).__name__,
                "error": str(exc),
            })
            print(f"  LOAD ERROR: {type(exc).__name__}: {exc}")
            continue

        for repeat_index, seed in enumerate(repeat_seeds):
            try:
                row, obs = run_one_repeat(
                    path.name,
                    X_raw,
                    y,
                    seed,
                    repeat_index,
                )
                run_rows.append(row)

                if obs is not None:
                    observation_parts.append(obs)

                print(
                    f"  repeat {repeat_index + 1:02d}/{N_REPEATS}: "
                    f"rank shift={row['reference_mean_abs_rank_shift']:.4f}, "
                    f"top10 overlap={row['reference_top10_overlap']:.3f}, "
                    f"held-out max |score diff|="
                    f"{row['heldout_max_abs_score_diff']:.3e}"
                )

            except Exception as exc:
                errors.append({
                    "dataset": path.name,
                    "repeat": int(repeat_index),
                    "seed": int(seed),
                    "stage": "experiment",
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                })
                print(
                    f"  RUN ERROR repeat {repeat_index}: "
                    f"{type(exc).__name__}: {exc}"
                )

    audit_df = pd.DataFrame(loader_audit)
    seed_df = pd.DataFrame(run_rows)
    error_df = pd.DataFrame(errors)

    audit_df.to_csv(
        OUTPUT_DIR / "table1_regime_loader_audit.csv",
        index=False,
    )
    error_df.to_csv(
        OUTPUT_DIR / "table1_regime_errors.csv",
        index=False,
    )

    if seed_df.empty:
        print("\nNo experiment run completed. The first recorded errors are:")
        if not error_df.empty:
            cols = [
                c for c in
                ["dataset", "repeat", "seed", "stage", "error_type", "error"]
                if c in error_df.columns
            ]
            print(error_df[cols].head(20).to_string(index=False))
        else:
            print("No detailed error rows were recorded.")

        raise RuntimeError(
            "No experiment runs completed. The underlying errors are printed "
            "above and saved in table1_regime_errors.csv."
        )

    seed_df.to_csv(
        OUTPUT_DIR / "table1_regime_seed_level.csv",
        index=False,
    )

    if observation_parts:
        obs_df = pd.concat(observation_parts, ignore_index=True)
        obs_df.to_csv(
            OUTPUT_DIR / "table1_regime_observation_sample.csv",
            index=False,
        )

    ds_df = dataset_summary(seed_df)
    ds_df.to_csv(
        OUTPUT_DIR / "table1_regime_dataset_summary.csv",
        index=False,
    )

    overall_df = overall_summary(seed_df, ds_df)
    overall_df.to_csv(
        OUTPUT_DIR / "table1_regime_overall_summary.csv",
        index=False,
    )

    make_figures(ds_df)

    config = {
        "data_dir": str(DATA_DIR.resolve()),
        "output_dir": str(OUTPUT_DIR.resolve()),
        "expected_file_count": EXPECTED_FILE_COUNT,
        "n_repeats": N_REPEATS,
        "n_trees": N_TREES,
        "test_size": TEST_SIZE,
        "base_seed": BASE_SEED,
        "repeat_seeds": repeat_seeds,
        "psi_cap": PSI_CAP,
        "psi_fraction": PSI_FRACTION,
        "top_fractions": list(TOP_FRACTIONS),
        "rank_shift_thresholds": list(RANK_SHIFT_THRESHOLDS),
        "equality_atol": EQUALITY_ATOL,
    }

    with open(
        OUTPUT_DIR / "table1_regime_experiment_config.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(config, f, indent=2, default=json_default)

    print("\nCompleted.")
    print("\nOverall regime illustration:")
    print(overall_df.to_string(index=False))

    failed_files = sorted(error_df["dataset"].unique()) if not error_df.empty else []
    if failed_files:
        print("\nFiles with at least one recorded error:")
        for name in failed_files:
            print("  -", name)

    print(f"\nOutputs written to: {OUTPUT_DIR.resolve()}")

    return {
        "loader_audit": audit_df,
        "seed_level": seed_df,
        "dataset_summary": ds_df,
        "overall_summary": overall_df,
        "errors": error_df,
    }


# In a notebook, run `results = main()` from the final cell.

## Preflight check

Run this before the full experiment. It loads the first dataset and executes one repeat. If anything fails, the actual exception appears immediately.


In [12]:
_files = sorted(
    [
        p for p in DATA_DIR.iterdir()
        if p.is_file() and p.suffix.lower() in {".arff", ".mat"}
    ],
    key=lambda p: p.name.lower(),
)

print(f"Found {len(_files)} dataset files in: {DATA_DIR.resolve()}")

if not _files:
    raise RuntimeError("No .arff/.mat files found. Check DATA_DIR.")

_test_path = _files[0]
print(f"Preflight dataset: {_test_path.name}")

_X_raw, _y, _audit = load_dataset(_test_path)

print("\nLoader audit:")
print(pd.DataFrame([_audit]).to_string(index=False))

_test_seed = int(np.random.SeedSequence(BASE_SEED).generate_state(1)[0])

_test_row, _ = run_one_repeat(
    _test_path.name,
    _X_raw,
    _y,
    _test_seed,
    0,
)

print("\nPreflight succeeded.")
print(
    pd.Series({
        "reference_mean_abs_rank_shift":
            _test_row["reference_mean_abs_rank_shift"],
        "reference_top10_overlap":
            _test_row["reference_top10_overlap"],
        "heldout_max_abs_score_diff":
            _test_row["heldout_max_abs_score_diff"],
        "heldout_exact_score_equal":
            _test_row["heldout_exact_score_equal"],
    }).to_string()
)


Found 30 dataset files in: C:\Users\jzhou\Desktop\Academics\Research\Outlier Detection\OOBMeanPath\Refined and subject to sumit version\Table 1 Verification
Preflight dataset: Annthyroid_withoutdupl_07.arff

Loader audit:
                       dataset    n         raw_class_counts                anomaly_rule  n_anomalies  anomaly_fraction                feature_source target_source
Annthyroid_withoutdupl_07.arff 7129 {"no": 6595, "yes": 534} semantic anomaly label: yes          534          0.074905 ARFF attributes except target       outlier

Preflight succeeded.
reference_mean_abs_rank_shift    0.004138
reference_top10_overlap          0.985989
heldout_max_abs_score_diff            0.0
heldout_exact_score_equal            True


## Run the full experiment

Run this only after the preflight succeeds.


In [13]:
results = main()

print("\nDataset-level summary:")
display(results["dataset_summary"])

print("\nOverall summary:")
display(results["overall_summary"])

if not results["errors"].empty:
    print("\nRecorded errors:")
    display(results["errors"])


Environment:
  NumPy:        1.26.4
  pandas:       2.2.2
  SciPy:        1.13.1
  scikit-learn: 1.4.2

Discovered 30 dataset files.
Repeats per dataset: 20
Trees per forest: 500

[01/30] Annthyroid_withoutdupl_07.arff
  repeat 01/20: rank shift=0.0041, top10 overlap=0.986, held-out max |score diff|=0.000e+00
  repeat 02/20: rank shift=0.0041, top10 overlap=0.986, held-out max |score diff|=0.000e+00
  repeat 03/20: rank shift=0.0043, top10 overlap=0.989, held-out max |score diff|=0.000e+00
  repeat 04/20: rank shift=0.0040, top10 overlap=0.982, held-out max |score diff|=0.000e+00
  repeat 05/20: rank shift=0.0041, top10 overlap=0.991, held-out max |score diff|=0.000e+00
  repeat 06/20: rank shift=0.0041, top10 overlap=0.986, held-out max |score diff|=0.000e+00
  repeat 07/20: rank shift=0.0041, top10 overlap=0.993, held-out max |score diff|=0.000e+00
  repeat 08/20: rank shift=0.0039, top10 overlap=0.986, held-out max |score diff|=0.000e+00
  repeat 09/20: rank shift=0.0041, top10 over

,dataset,repeats,n_total,n_reference_mean,n_heldout_mean,d_processed_mean,psi_mean,q_oob_mean,reference_score_spearman_mean,reference_mean_abs_rank_shift_mean,...,reference_p95_abs_C_over_cpsi_mean,heldout_max_abs_path_diff_max,heldout_max_abs_score_diff_max,heldout_exact_path_equal_all_repeats,heldout_exact_score_equal_all_repeats,heldout_equal_within_tolerance_all_repeats,reference_delta_ROC_mean,reference_delta_AP_mean,heldout_delta_ROC_max_abs,heldout_delta_AP_max_abs
0,Annthyroid_withoutdupl_07.arff,20,7129,5703.0,1426.0,22.0,256.0,0.955111,0.999821,0.004091,...,0.006569,0.0,0.0,True,True,True,-0.000069,0.000084,0.0,0.0
1,Cardiotocography_22.arff,20,2126,1700.0,426.0,22.0,256.0,0.849412,0.998695,0.010902,...,0.012335,0.0,0.0,True,True,True,0.000077,0.000747,0.0,0.0
2,Cardiotocography_withoutdupl_norm_10_v10.arff,20,1831,1464.0,367.0,22.0,256.0,0.825137,0.998507,0.011674,...,0.013501,0.0,0.0,True,True,True,-0.000404,0.000581,0.0,0.0
3,HeartDisease_withoutdupl_44.arff,20,270,216.0,54.0,14.0,172.0,0.203704,0.974471,0.048150,...,0.063392,0.0,0.0,True,True,True,0.003533,0.002614,0.0,0.0
4,InternetAds_withoutdupl_norm_19.arff,20,1966,1572.0,394.0,1556.0,256.0,0.837150,0.992504,0.025454,...,0.014677,0.0,0.0,True,True,True,0.001784,-0.005986,0.0,0.0
5,KDDCup99_original.arff,20,60839,48671.0,12168.0,77.7,256.0,0.994740,0.999960,0.001629,...,0.001969,0.0,0.0,True,True,True,-0.000058,-0.002401,0.0,0.0
6,PageBlocks_withoutdupl_norm_05_v10.arff,20,5139,4111.0,1028.0,11.0,256.0,0.937728,0.999683,0.005165,...,0.006506,0.0,0.0,True,True,True,-0.000211,-0.000926,0.0,0.0
7,SpamBase_withoutdupl_norm_10_v10.arff,20,2808,2246.0,562.0,58.0,256.0,0.886020,0.998507,0.011283,...,0.011346,0.0,0.0,True,True,True,-0.000583,-0.000507,0.0,0.0
8,arrhythmia.arff,20,452,361.0,91.0,340.0,256.0,0.290859,0.967952,0.052922,...,0.056721,0.0,0.0,True,True,True,-0.000883,0.001163,0.0,0.0
9,breastw.mat,20,683,546.0,137.0,9.0,256.0,0.531136,0.997372,0.013888,...,0.030603,0.0,0.0,True,True,True,0.001016,0.003016,0.0,0.0



Overall summary:


,n_datasets,n_dataset_repeat_runs,n_trees,n_repeats,test_size,reference_dataset_median_of_mean_abs_rank_shift,reference_dataset_mean_of_mean_abs_rank_shift,reference_dataset_median_top10_overlap,reference_dataset_median_fraction_rank_shift_ge_5pct,reference_dataset_median_rho_absC_rankshift,heldout_global_max_abs_path_diff,heldout_global_max_abs_score_diff,heldout_exact_path_equal_all_runs,heldout_exact_score_equal_all_runs,heldout_equal_within_tolerance_all_runs,reference_delta_ROC_across_datasets_mean,reference_delta_AP_across_datasets_mean
0,30,600,500,20,0.2,0.011569,0.017019,0.974099,0.007526,0.565657,0.0,0.0,True,True,True,-0.00042,-0.001634
